# Task 2 — Library Dataset Management
**Course:** STQD6014 Data Science  
**Files:** `library.json` (source) → `library_updated.json` (output)

**Tasks:**
1. Load `library.json` into a pandas DataFrame
2. Search for books by genre
3. Update availability status of a book
4. Add a new book entry
5. Save the final dataset to `library_updated.json`

**FIX:** The original notebook called `add_new_book()` twice (once as a print/display
and once to save), resulting in B006 appearing twice as a duplicate row in
`library_updated.json`. Fixed by calling it once and saving that result.

In [ ]:
import pandas as pd
import json

# Load original library dataset
df = pd.read_json('data/library.json')
print(f"Library loaded: {df.shape[0]} books")
df

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def search_by_genre(df: pd.DataFrame, genre: str) -> pd.DataFrame:
    """Return all books in the given genre (case-sensitive)."""
    return df[df['Genre'] == genre].reset_index(drop=True)


def update_availability(df: pd.DataFrame, book_id: str,
                        new_status: bool) -> pd.DataFrame:
    """Set Availability for the given Book_ID and return updated DataFrame."""
    df = df.copy()
    if book_id not in df['Book_ID'].values:
        print(f"  Warning: Book_ID '{book_id}' not found.")
        return df
    df.loc[df['Book_ID'] == book_id, 'Availability'] = new_status
    return df


def add_new_book(df: pd.DataFrame, new_book: dict) -> pd.DataFrame:
    """Append one new book record to the DataFrame and return the result."""
    # FIX: use drop_duplicates to guard against calling this function twice
    df_new = pd.concat([df, pd.DataFrame([new_book])], ignore_index=True)
    df_new = df_new.drop_duplicates(subset=['Book_ID', 'Title'], keep='first')
    return df_new.reset_index(drop=True)

## Operation 1 — Search by genre

In [ ]:
result = search_by_genre(df, 'Drama')
print(f"Books in 'Drama' genre ({len(result)} found):")
result

## Operation 2 — Update availability

In [ ]:
# B002 was originally False; update to True
df_updated = update_availability(df, 'B002', True)
print("B002 availability after update:", df_updated.loc[df_updated['Book_ID'] == 'B002', 'Availability'].values[0])
df_updated

## Operation 3 — Add a new book

In [ ]:
new_book = {
    "Book_ID": "B006",
    "Title": "Ocean of Dreams",
    "Author": "Hassan Malik",
    "Year_of_Publication": 2023,
    "Genre": "Fantasy",
    "Availability": True
}

# FIX: call add_new_book only once; original called it twice → duplicate row B006
df_final = add_new_book(df_updated, new_book)
print(f"Library after adding B006: {df_final.shape[0]} books (no duplicates)")
df_final

## Save to library_updated.json

In [ ]:
# Save final dataset — note: orient='records' gives a cleaner list-of-objects format
df_final.to_json('data/library_updated.json', orient='records', indent=4)
print("Saved: data/library_updated.json")
print()

# Verify by reading back
verify = pd.read_json('data/library_updated.json')
print(f"Verification — rows: {len(verify)}, duplicates: {verify.duplicated().sum()}")
verify